# The Bernstein-Vazirani Algorithm

## An elegant variation of Deutsch–Jozsa: the oracle hides a secret bit string $s \in \{0,1\}^n$, and you have to recover it. Classically: $n$ queries. Quantumly: **one**.

# 1. The problem

## An oracle implements $f : \{0,1\}^n \to \{0,1\}$ of the form

$$ \Large  f(x) = s \cdot x \bmod 2 \;=\; s_1 x_1 \oplus s_2 x_2 \oplus \cdots \oplus s_n x_n $$

## for some unknown $s$. **Task**: determine $s$.

## **Classically**, query at all $n$ standard basis vectors $e_1, \dots, e_n$ to read off each bit of $s$ — that's $n$ queries. **Quantumly**, one application of the oracle is enough.

# 2. The algorithm

## Identical to Deutsch–Jozsa:

## 1. Initialise $|0\rangle^{\otimes n}|1\rangle$.
## 2. Apply $H$ to all $n + 1$ qubits.
## 3. Apply $U_f$.
## 4. Apply $H^{\otimes n}$ to the input register.
## 5. Measure the input register.

## The measured bitstring is exactly $s$. Deterministic, in one query.

# 3. Why it works

## After step 3 the input register is

$$ \Large  \frac{1}{\sqrt{2^n}} \sum_x (-1)^{s \cdot x} |x\rangle. $$

## Applying $H^{\otimes n}$ uses the Walsh–Hadamard identity

$$ \Large  H^{\otimes n} \sum_x (-1)^{s \cdot x} |x\rangle = \sqrt{2^n}\, |s\rangle. $$

## So the state collapses exactly to $|s\rangle$ — measurement reveals $s$ with certainty.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def bv_oracle(s):
    n = len(s)
    qc = QuantumCircuit(n + 1)
    for i, bit in enumerate(reversed(s)):     # qiskit little-endian
        if bit == '1':
            qc.cx(i, n)
    return qc

def bernstein_vazirani(s):
    n = len(s)
    qc = QuantumCircuit(n + 1, n)
    qc.x(n); qc.h(n)
    qc.h(range(n))
    qc.compose(bv_oracle(s), inplace=True)
    qc.h(range(n))
    qc.measure(range(n), range(n))
    sim = AerSimulator()
    counts = sim.run(transpile(qc, sim), shots=1).result().get_counts()
    return next(iter(counts))

secret = '1011010'
print(f'secret = {secret}, recovered = {bernstein_vazirani(secret)}')

# 4. What's really going on

## Bernstein–Vazirani is a special case of **quantum Fourier sampling** over the group $(\mathbb{Z}/2)^n$. The Hadamard transform is exactly the QFT over that group, and the algorithm just samples the Fourier coefficient — which concentrates entirely at $s$. This is a foreshadowing of Shor's algorithm, where the same idea is used over $\mathbb{Z}/N$ to extract the period of a function.

# Recap

## - Oracle: $f(x) = s \cdot x \bmod 2$. Recover $s$.
## - Classical: $n$ queries. Quantum: 1.
## - Same circuit shape as Deutsch–Jozsa, but the output bitstring is now the answer.
## - A small but clear example of Fourier sampling, the engine of Shor.